# Recursive Polynomial Basis — v1

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ---------- Recursive polynomial basis ----------

class RecursivePolyBasis(nn.Module):
    """
    R_0(x) = 0
    R_1(x) = 1
    R_{n+1}(x) = (a x^2 + b x + c) R_n(x) + (d x + e) R_{n-1}(x)
    y = sum_{n=0}^K w_n R_n(x)
    """
    def __init__(self, K, init_like_cheb=True, param_bound=3.0):
        super().__init__()
        self.K = K
        self.param_bound = param_bound

        # unconstrained parameters (we will squash with tanh)
        self.a_raw = nn.Parameter(torch.zeros(1))
        self.b_raw = nn.Parameter(torch.zeros(1))
        self.c_raw = nn.Parameter(torch.zeros(1))
        self.d_raw = nn.Parameter(torch.zeros(1))
        self.e_raw = nn.Parameter(torch.zeros(1))

        # weights for basis
        self.w = nn.Parameter(torch.zeros(K + 1))

        if init_like_cheb:
            with torch.no_grad():
                # approximate Chebyshev-like: 2x * R_n - R_{n-1}
                self.a_raw.fill_(0.0)
                # tanh^-1(2/param_bound) but keep small to avoid saturation
                self.b_raw.fill_(0.5)
                self.c_raw.fill_(0.0)
                self.d_raw.fill_(0.0)
                self.e_raw.fill_(-0.5)
                self.w.normal_(mean=0.0, std=0.01)

    def _squash_params(self):
        # map raw params to bounded range via tanh for stability
        a = self.param_bound * torch.tanh(self.a_raw)
        b = self.param_bound * torch.tanh(self.b_raw)
        c = self.param_bound * torch.tanh(self.c_raw)
        d = self.param_bound * torch.tanh(self.d_raw)
        e = self.param_bound * torch.tanh(self.e_raw)
        return a, b, c, d, e

    def forward(self, x):
        """
        x: tensor of shape (batch,) or (batch,1)
        returns: (batch,)
        """
        if x.dim() > 1:
            x = x.squeeze(-1)

        # assume x already in a reasonable range; optional clip
        x = x.clamp(-2.0, 2.0)

        a, b, c, d, e = self._squash_params()

        R0 = torch.zeros_like(x)
        R1 = torch.ones_like(x)

        Rs = [R0, R1]

        for n in range(1, self.K):
            coef1 = a * x**2 + b * x + c
            coef2 = d * x + e
            R_next = coef1 * Rs[-1] + coef2 * Rs[-2]
            Rs.append(R_next)

        R_stack = torch.stack(Rs, dim=-1)  # (batch, K+1)
        y = (R_stack * self.w).sum(dim=-1)
        return y

# ---------- Simple model: linear in -> recursive basis out ----------

class PolyKAN1D(nn.Module):
    def __init__(self, K):
        super().__init__()
        # one linear projection (can be identity, but we keep flexibility)
        self.lin = nn.Linear(1, 1)
        self.basis = RecursivePolyBasis(K=K, init_like_cheb=True)

    def forward(self, x):
        # x: (batch,1)
        h = self.lin(x)           # (batch,1)
        h = h.squeeze(-1)         # (batch,)
        y = self.basis(h)         # (batch,)
        return y.unsqueeze(-1)    # (batch,1)

# ---------- Target function f(x) = x^2 + sin(x) ----------

def target_function(x):
    return x**2 + torch.sin(x)

# ---------- Training loop ----------

def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # hyperparameters
    K = 8            # number of basis functions - 1 (we get R_0..R_K)
    n_samples = 2000
    n_epochs = 2000
    batch_size = 256
    lr = 1e-3

    # data: uniform on [-2, 2]
    x_all = (4.0 * torch.rand(n_samples, 1) - 2.0).to(device)
    y_all = target_function(x_all).to(device)

    model = PolyKAN1D(K=K).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(1, n_epochs + 1):
        perm = torch.randperm(n_samples)
        x_all = x_all[perm]
        y_all = y_all[perm]

        for i in range(0, n_samples, batch_size):
            x_batch = x_all[i:i+batch_size]
            y_batch = y_all[i:i+batch_size]

            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            print(f"Epoch {epoch}/{n_epochs}, loss = {loss.item():.6f}")

    # evaluate on a fine grid for plotting
    x_test = torch.linspace(-2, 2, 400).unsqueeze(-1).to(device)
    with torch.no_grad():
        y_pred = model(x_test)
    y_true = target_function(x_test)

    x_np = x_test.cpu().numpy().squeeze()
    y_pred_np = y_pred.cpu().numpy().squeeze()
    y_true_np = y_true.cpu().numpy().squeeze()

    plt.figure(figsize=(6,4))
    plt.plot(x_np, y_true_np, label='x^2 + sin(x)', color='black')
    plt.plot(x_np, y_pred_np, label='PolyKAN approx', color='red', linestyle='--')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    train_model()
